In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dataclasses import asdict, dataclass

from areal.api.agent_args import AgentGRPOConfig, load_expr_config

args = ["--config", "AReaL/examples/lite/configs/sokoban_grpo_vision.yaml"]
config, _ = load_expr_config(args, AgentGRPOConfig)
config: AgentGRPOConfig

/home/aiscuser/.conda/envs/areal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-14 12:22:24,319	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
config

AgentGRPOConfig(experiment_name='areal', trial_name='sokoban-grpo-vision', cluster=ClusterSpecConfig(name_resolve=NameResolveConfig(type='nfs', nfs_record_root='/tmp/areal/name_resolve', etcd3_addr='localhost:2379', ray_actor_name='ray_kv_store'), cluster_name='local', fileroot='/tmp/areal/experiments', n_nodes=1, n_gpus_per_node=8), allocation_mode='sglang.d4p1t1+d4p1t1', seed=1, total_train_epochs=10, total_train_steps=None, total_train_n_seqs=None, tokenizer_path='Qwen/Qwen2.5-VL-3B-Instruct', train_dataset=DatasetConfig(path='openai/gsm8k', type='rl', batch_size=16, shuffle=True, pin_memory=True, num_workers=4, drop_last=True, reward_fn=None), valid_dataset=DatasetConfig(path='openai/gsm8k', type='rl', batch_size=256, shuffle=True, pin_memory=True, num_workers=4, drop_last=True, reward_fn=None), saver=SaverConfig(experiment_name='areal', trial_name='sokoban-grpo-vision', fileroot='/tmp/areal/experiments', freq_epochs=1, freq_steps=None, freq_secs=None), evaluator=EvaluatorConfig(ex

In [4]:
from areal.utils.network import find_free_ports

SGLANG_PORT, MASTER_PORT = 11451, 14514

SGLANG_HOST = "127.0.0.1"

# Environment variables used by inference/train engines
import os

os.environ["AREAL_LLM_SERVER_ADDRS"] = f"{SGLANG_HOST}:{SGLANG_PORT}"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = str(MASTER_PORT)
os.environ["RANK"] = str(0)
os.environ["WORLD_SIZE"] = str(1)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["LOCAL_RANK"] = str(0)

In [5]:
import subprocess
import sys

# 启动sglang server
from areal.api.cli_args import SGLangConfig
from areal.utils.network import find_free_ports

config.sglang.log_level = "info"
config.sglang.decode_log_interval = 10
sglang_cmd = SGLangConfig.build_cmd(
    config.sglang,
    tp_size=1,
    base_gpu_id=1,
    host=SGLANG_HOST,
    port=SGLANG_PORT,
)
sglang_process = subprocess.Popen(
    sglang_cmd,
    shell=True,
    stdout=sys.stdout,
    stderr=sys.stderr,
)

In [6]:
import asyncio
import functools
import os
import time
import uuid

import colorama
import torch
from tensordict import TensorDict
from transformers import AutoTokenizer, PreTrainedTokenizerFast

from areal.api.cli_args import GenerationHyperparameters
from areal.api.engine_api import InferenceEngine
from areal.api.io_struct import (
    AllocationMode,
    FinetuneSpec,
    WeightUpdateMeta,
)
from areal.api.workflow_api import RolloutWorkflow
from areal.engine.ppo.actor import FSDPPPOActor
from areal.engine.sglang_remote import RemoteSGLangEngine
from areal.utils.data import concat_padded_tensors
from areal.utils.device import log_gpu_stats
from realhf.api.core.data_api import load_hf_processor_and_tokenizer



In [7]:
processor, tokenizer = load_hf_processor_and_tokenizer(config.tokenizer_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [8]:
config.envs

[EnvSpec(name='sokoban', n_envs=10000, split='train', seed=0, config={'render_mode': 'vision', 'dim_room': '(6, 6)', 'num_boxes': 1}),
 EnvSpec(name='sokoban', n_envs=5000, split='train', seed=0, config={'render_mode': 'vision', 'dim_room': '(8, 8)', 'num_boxes': 1}),
 EnvSpec(name='sokoban', n_envs=128, split='valid', seed=0, config={'render_mode': 'vision', 'dim_room': '(6, 6)', 'num_boxes': 1}),
 EnvSpec(name='sokoban', n_envs=64, split='valid', seed=0, config={'render_mode': 'vision', 'dim_room': '(8, 8)', 'num_boxes': 1})]

In [9]:
from torchdata.stateful_dataloader import StatefulDataLoader
from areal.dataset.multi_env_dataset import build_env_dataset
rank=0
world_size = 1
train_dataset = build_env_dataset(
        config.envs, split="train", base_seed=config.seed, rank=rank, world_size=world_size
    )
dataloader = StatefulDataLoader(
        train_dataset,
        batch_size=config.train_dataset.batch_size // world_size,
        shuffle=config.train_dataset.shuffle,
        num_workers=config.train_dataset.num_workers,
        collate_fn=lambda x: x,
        drop_last=config.train_dataset.drop_last,
    )


from itertools import cycle

data_generator = cycle(dataloader)

ft_spec = FinetuneSpec(
    total_train_epochs=config.total_train_epochs,
    dataset_size=len(dataloader) * config.train_dataset.batch_size,
    train_batch_size=config.train_dataset.batch_size,
)

x = next(data_generator)


In [10]:
print(x[0])
print(x[0].keys())

{'name': 'sokoban', 'seed': 2093497556, 'config': {'render_mode': 'vision', 'dim_room': '(6, 6)', 'num_boxes': 1}}
dict_keys(['name', 'seed', 'config'])


In [11]:
os.environ["AREAL_DEBUG_TOKEN_ALIGN"] = "1"

In [12]:
# initialize inference engine
from areal.engine.sglang_remote import RemoteSGLangEngine
from areal.workflow.vision_multi_turn_agent import VisionMultiTurnAgentEnvWorkflow
rollout = RemoteSGLangEngine(config.rollout)
rollout.initialize(None, None)
try:
    # TODO: create workflow
    workflow = VisionMultiTurnAgentEnvWorkflow(
        gconfig=GenerationHyperparameters(n_samples=3,max_new_tokens=512),
        tokenizer=tokenizer,
        max_turns=3,
        processor=processor,
        dump_dir="./test"
    )
    sample_data = next(data_generator)[:2]
    res = rollout.rollout_batch(sample_data, workflow=workflow)
    print(res)
finally:
    rollout.destroy()

20250814-12:22:35.397 areal.engine.sglang_remote INFO: Waiting for server ready...
20250814-12:22:53.433 areal.engine.sglang_remote INFO: Servers are all ready!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


torch.Size([1, 144, 1176]) torch.Size([1, 4, 3])
torch.Size([1, 144, 1176]) torch.Size([1, 4, 3])
torch.Size([1, 144, 1176]) torch.Size([1, 4, 3])
torch.Size([1, 144, 1176]) torch.Size([1, 4, 3])
torch.Size([1, 108, 1176]) torch.Size([1, 3, 3])
torch.Size([1, 144, 1176]) torch.Size([1, 4, 3])
TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([6, 772]), device=cpu, dtype=torch.bool, is_shared=False),
        image_grid_thw: Tensor(shape=torch.Size([6, 4, 3]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([6, 772]), device=cpu, dtype=torch.int64, is_shared=False),
        logprobs: Tensor(shape=torch.Size([6, 772]), device=cpu, dtype=torch.float32, is_shared=False),
        loss_mask: Tensor(shape=torch.Size([6, 772]), device=cpu, dtype=torch.int64, is_shared=False),
        pixel_values: Tensor(shape=torch.Size([6, 144, 1176]), device=cpu, dtype=torch.float32, is_shared=False),
        rewards: Tensor(shape=torch.Size(